# DFU-ImageGuard — Complete Conference Experiment

Run the single code cell below. It installs non-framework dependencies without replacing Colab's CUDA-enabled PyTorch, refreshes the repository, clears stale imported modules, verifies nested Drive writes, resumes interrupted folds, and executes the leakage-safe five-fold experiment. No output is prefilled.


In [ ]:
import os, sys, subprocess, importlib, shutil
from pathlib import Path

os.environ.setdefault("PYTHONUNBUFFERED", "1")
os.environ.setdefault("HF_HUB_DISABLE_TELEMETRY", "1")

PACKAGES = [
    "timm>=1.0.9,<2",
    "ImageHash>=4.3,<5",
    "kagglehub>=0.3,<1",
    "joblib>=1.4,<2",
    "shap>=0.46,<1",
    "lime>=0.2.0.1,<1",
    "grad-cam>=1.5,<2",
    "opencv-python-headless>=4.10,<5",
    "pyyaml>=6,<7",
]
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "--disable-pip-version-check", "--upgrade-strategy", "only-if-needed",
    *PACKAGES,
])

try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
    if hf_token:
        os.environ["HF_TOKEN"] = hf_token
except Exception:
    pass

REPO = Path("/content/DFU-ImageGuard")
if (REPO / ".git").exists():
    subprocess.run(["git", "-C", str(REPO), "fetch", "origin", "main"], check=True)
    subprocess.run(["git", "-C", str(REPO), "reset", "--hard", "origin/main"], check=True)
else:
    if REPO.exists():
        shutil.rmtree(REPO)
    subprocess.check_call([
        "git", "clone", "https://github.com/AzizulHakim00/DFU-ImageGuard.git", str(REPO)
    ])

for module_name in list(sys.modules):
    if module_name == "src" or module_name.startswith("src."):
        del sys.modules[module_name]
importlib.invalidate_caches()
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

repo_commit = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
print(f"DFU-ImageGuard repository commit: {repo_commit}")

from src.config_data import Config as PreflightConfig, mount_drive
from src.runtime_io import storage_write_probe
preflight = PreflightConfig()
mount_drive(preflight)
print("Storage preflight:", storage_write_probe(preflight.DRIVE_ROOT))

from src.dfu_imageguard_pipeline import run_complete_pipeline

OVERRIDES = {
    "LOCAL_REPO": str(REPO),
    "FORCE_RETRAIN": False,
    "RUN_BASELINES": True,
    "RUN_ROBUSTNESS": True,
    "RUN_XAI": True,
    "MAX_EPOCHS": 30,
    "PATIENCE": 7,
    "BOOTSTRAP_REPS": 1000,
    "NUM_WORKERS": 2,
}
FINAL_REPORT = run_complete_pipeline(OVERRIDES)
FINAL_REPORT
